# Partitura `<annot>` Failure and CAMAT Fix

This notebook demonstrates two things on the same MEI file:

1. Raw `partitura` parsing fails when `<annot>` tags are present.
2. `camat.partitura_backend.parse_files_partitura` succeeds by using the current preprocessing + retry pipeline (while still staying on the partitura backend).


In [ ]:
from pathlib import Path
import re
import pandas as pd

# Local MEI with annotation tags (no network required)
MEI_PATH = Path('CAMAT_revamped/exports/score_with_annot_20251117_105451.mei')
print('File exists:', MEI_PATH.exists())
print('Path:', MEI_PATH)

txt = MEI_PATH.read_text(encoding='utf-8', errors='ignore')
print('Count <annot> tags:', len(re.findall(r'<annot\b', txt)))


## 1) Raw partitura behavior

This should fail with `element ... annot is not yet supported`.


In [ ]:
import partitura as pt

try:
    score = pt.load_score(str(MEI_PATH))
    print('UNEXPECTED: raw partitura parse succeeded. Parts:', len(score.parts))
except Exception as exc:
    print('Expected raw partitura failure:')
    print(type(exc).__name__, str(exc))


## 2) Current CAMAT partitura pipeline

Use strict partitura-only mode (`allow_music21_fallback=False`) to prove this is not using music21 fallback.


In [ ]:
from camat.partitura_backend import parse_files_partitura

results, dfs_by_name, last_df = parse_files_partitura(
    [str(MEI_PATH)],
    backend='none',
    display_preview=False,
    show_progress=False,
    normalize_mensural_durations=True,
    inject_missing_meter_signature=True,
    default_meter_count=2,
    default_meter_unit=2,
    try_verovio_mei_conversion=True,
    verovio_mensural_to_cmn=True,
    allow_music21_fallback=False,
)

print('Parsed entries:', len(results))
print('DataFrames keys:', list(dfs_by_name.keys()))
print('last_df is None:', last_df is None)
if last_df is not None:
    print('Rows:', len(last_df), 'Columns:', list(last_df.columns))
    display(last_df.head(10))


## 3) Optional: inspect conversion/postprocess stats

This cell uses internal helpers only for debug/demo purposes.


In [ ]:
from camat.partitura_backend import _convert_mei_with_verovio_for_partitura

converted_path, cleanup_fn, removed_annots, wrapped_staff_groups = _convert_mei_with_verovio_for_partitura(
    str(MEI_PATH),
    mensural_to_cmn=True,
)

print('Converted temp path:', converted_path)
print('Removed <annot> count:', removed_annots)
print('Wrapped section-level staff groups:', wrapped_staff_groups)

tmp_txt = Path(converted_path).read_text(encoding='utf-8', errors='ignore')
print('Remaining <annot> tags in converted temp MEI:', len(re.findall(r'<annot\b', tmp_txt)))

if cleanup_fn:
    cleanup_fn()
